In [6]:
# 导入必要的库
import metpy.calc as mpcalc
from metpy.units import units
import gc  # 垃圾回收
import os
import shutil
from pathlib import Path
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
# Set data path
data_path = Path('/work/mh1498/m301257/')
fig_save_dir = "./figures/Cal_temperature_profile/"
os.makedirs(fig_save_dir, exist_ok=True)

def get_dir_size(path):
    """计算目录大小（GB）"""
    total = 0
    try:
        for entry in os.scandir(path):
            if entry.is_file():
                total += entry.stat().st_size
            elif entry.is_dir():
                total += get_dir_size(entry.path)
    except:
        pass
    return total / (1024**3)  # 转换为GB


def _ocean(ds):
    fraction = xr.open_dataarray(r'./processed_data/land_mask_2deg.nc')
    return fraction == 0

# 检查缓存目录
cache_dirs = {
    'cache': Path('./cache'),
    'dask-worker-space': Path('./dask-worker-space'),
    'dask-scratch-space': Path('./dask-scratch-space')
}

print("="*80)
print("当前缓存状态")
print("="*80)
for name, path in cache_dirs.items():
    if path.exists():
        size = get_dir_size(path)
        print(f"{name:.<30} {size:>8.2f} GB")
print("="*80)

# 清理选项
CLEAN_DASK = True  # 清理Dask临时文件
CLEAN_OLD_CACHE = False  # 是否清理旧缓存（谨慎使用！）

if CLEAN_DASK:
    print("\n清理Dask临时文件...")
    for name in ['dask-worker-space', 'dask-scratch-space']:
        path = cache_dirs[name]
        if path.exists():
            try:
                shutil.rmtree(path)
                path.mkdir(exist_ok=True)
                print(f"  ✓ 已清理: {name}")
            except Exception as e:
                print(f"  ✗ 清理失败: {name} - {e}")

if CLEAN_OLD_CACHE:
    print("\n⚠️ 清理数据缓存（此操作会删除所有缓存文件）...")
    cache_path = cache_dirs['cache']
    if cache_path.exists():
        response = input("确定要删除所有缓存吗？这将需要重新计算数据。(yes/no): ")
        if response.lower() == 'yes':
            shutil.rmtree(cache_path)
            cache_path.mkdir(exist_ok=True)
            print("  ✓ 缓存已清理")
        else:
            print("  ✗ 已取消")

print("\n" + "="*80)
print("清理后状态")
print("="*80)
for name, path in cache_dirs.items():
    if path.exists():
        size = get_dir_size(path)
        print(f"{name:.<30} {size:>8.2f} GB")
print("="*80)
print("\n✓ 清理完成！建议重启kernel以释放内存。")

当前缓存状态
cache.........................     0.00 GB
dask-worker-space.............     0.00 GB
dask-scratch-space............     0.00 GB

清理Dask临时文件...
  ✓ 已清理: dask-worker-space
  ✓ 已清理: dask-scratch-space

清理后状态
cache.........................     0.00 GB
dask-worker-space.............     0.00 GB
dask-scratch-space............     0.00 GB

✓ 清理完成！建议重启kernel以释放内存。


## 1. 数据读取与预处理 (Data Loading and Preprocessing)

In [7]:

# 热带区域定义
TROPICAL_BOUNDS = (-15, 15)  # 15°S - 15°N
G = 9.81  # 重力加速度 (m/s²)

xr.set_options(keep_attrs=True)  # 保留属性但不缓存数据
# 读取陆地mask (0=ocean, 1=land)
print("\n读取陆地mask...")
land_mask = xr.open_dataset(data_path / 'processed_data/land_mask_2deg.nc')
ocean_mask = land_mask['__xarray_dataarray_variable__'] == 0  # 海洋为True
print(f"✓ Ocean grid points: {ocean_mask.sum().values}/{ocean_mask.size}")

print("="*80)
print("内存优化设置")
print("="*80)
print("✓ 禁用xarray文件缓存")
print("✓ 启用垃圾回收")
print("="*80)
# 1. 加载 zg 数据（位势高度，单位：m）
print("读取 zg 数据...")
zg_cntl = xr.open_dataset('processed_data/zg_2deg_interp_cntl.nc')['zg']
zg_p4k = xr.open_dataset('processed_data/zg_2deg_interp_p4k.nc')['zg']
zg_4co2 = xr.open_dataset('processed_data/zg_2deg_interp_4co2.nc')['zg']

# 2. 应用海洋 mask（与之前相同的 ocean_mask）
zg_cntl_ocean = zg_cntl.where(ocean_mask)
zg_p4k_ocean = zg_p4k.where(ocean_mask)
zg_4co2_ocean = zg_4co2.where(ocean_mask)

# 3. 计算海洋平均高度（对每一层）
# zg 不随时间变化，所以只对空间平均
zg_cntl_mean = zg_cntl_ocean.mean(dim=['lat', 'lon'], skipna=True)
zg_p4k_mean = zg_p4k_ocean.mean(dim=['lat', 'lon'], skipna=True)
zg_4co2_mean = zg_4co2_ocean.mean(dim=['lat', 'lon'], skipna=True)

# 转换为 km 单位
zg_cntl_km = zg_cntl_mean / 1000.0
zg_p4k_km = zg_p4k_mean / 1000.0
zg_4co2_km = zg_4co2_mean / 1000.0

print(f"✓ zg 数据加载完成")
print(f"  高度范围: {zg_cntl_km.min().values:.2f} - {zg_cntl_km.max().values:.2f} km")
print(f"  层数: {len(zg_cntl_km)}")


# 读取温度数据 - 使用chunks参数避免一次性加载所有数据到内存
print("\n读取温度数据（分块加载）...")
ta_cntl = xr.open_dataset(data_path / 'ta_cntl_layers/ta_all_levels.nc', chunks={'time': 10}).where(ocean_mask)
ta_p4k = xr.open_dataset(data_path / 'ta_p4k_layers/ta_all_levels.nc', chunks={'time': 10}).where(ocean_mask)
ta_4co2 = xr.open_dataset(data_path / 'ta_4co2_layers/ta_all_levels.nc', chunks={'time': 10}).where(ocean_mask)

# 读取压力数据
print("读取压力数据（分块加载）...")
pfull_cntl = xr.open_dataset(data_path / 'pfull_cntl_layers/pfull_all_levels.nc', chunks={'time': 10}).where(ocean_mask)
pfull_p4k = xr.open_dataset(data_path / 'pfull_p4k_layers/pfull_all_levels.nc', chunks={'time': 10}).where(ocean_mask)
pfull_4co2 = xr.open_dataset(data_path / 'pfull_4co2_layers/pfull_all_levels.nc', chunks={'time': 10}).where(ocean_mask)

# 自动检测变量名和维度名
ta_var = 'ta' if 'ta' in ta_cntl else list(ta_cntl.data_vars)[0]
pfull_var = 'pfull' if 'pfull' in pfull_cntl else list(pfull_cntl.data_vars)[0]

lat_dim = 'lat' if 'lat' in ta_cntl[ta_var].dims else 'latitude'
lon_dim = 'lon' if 'lon' in ta_cntl[ta_var].dims else 'longitude'

# 检测垂直维度名称
if 'lev' in ta_cntl[ta_var].dims:
    lev_dim = 'lev'
elif 'plev' in ta_cntl[ta_var].dims:
    lev_dim = 'plev'
elif 'level' in ta_cntl[ta_var].dims:
    lev_dim = 'level'
else:
    lev_dim = [d for d in ta_cntl[ta_var].dims if d not in ['time', lat_dim, lon_dim]][0]

print(f"✓ 变量名: ta={ta_var}, pfull={pfull_var}")
print(f"✓ 维度名: lat={lat_dim}, lon={lon_dim}, lev={lev_dim}")
print(f"✓ 数据形状: {ta_cntl[ta_var].shape}")

# 立即执行垃圾回收
gc.collect()



读取陆地mask...
✓ Ocean grid points: 2089/2700
内存优化设置
✓ 禁用xarray文件缓存
✓ 启用垃圾回收
读取 zg 数据...
✓ zg 数据加载完成
  高度范围: 0.01 - 22.86 km
  层数: 22

读取温度数据（分块加载）...
读取压力数据（分块加载）...
✓ 变量名: ta=ta, pfull=pfull
✓ 维度名: lat=lat, lon=lon, lev=level
✓ 数据形状: (22, 5114, 15, 180)


98

In [8]:
lev_dim

'level'

In [ ]:
# 转换高度到km
height_km = zg_cntl_km / 1000.0
min_height_km = 0.0  # 最小高度 (km)
max_height_km = 30.0  # 最大高度 (km)
ta_data = ta_cntl[ta_var]
height_data = zg_cntl_ocean
level_full = 'level_full' 
# 创建高度掩码（只在指定范围内搜索）
height_mask = (height_km >= min_height_km) & (height_km <= max_height_km)
valid_levels = np.where(height_mask)[0]

if len(valid_levels) == 0:
    raise ValueError(f"没有层次在指定高度范围内 ({min_height_km}-{max_height_km} km)")


# 只选择有效高度范围的数据
ta_subset = ta_data.isel({lev_dim: valid_levels})
height_subset = height_data.isel({level_full: valid_levels})

trop_idx_subset = ta_subset.argmin(dim=lev_dim)




In [ ]:
trop_idx_absolute = xr.DataArray(
        valid_levels[trop_idx_subset.values],
        coords=trop_idx_subset.coords,
        dims=trop_idx_subset.dims
    )
trop_idx_absolute

/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/dask/array/reductions.py:784: RuntimeWarning: All-NaN slice encountered
  vals = func(x, axis=arg_axis, keepdims=True)


ValueError: All NaN slice encountered

: 

In [10]:


# def calculate_tropopause_from_temperature_minimum(ta_data, height_data, 
#                                                    min_height_km=5, max_height_km=25):
#     """
#     从4D温度场计算对流层顶（基于温度最小值）
    
#     Parameters:
#     -----------
#     ta_data : xr.DataArray
#         4D温度数据 (time, lev, lat, lon)
#     pfull_data : xr.DataArray
#         4D气压数据 (time, lev, lat, lon)
#     height_data : xr.DataArray
#         1D高度数据 (lev), 单位: m
#     min_height_km : float
#         最小搜索高度 (km)
#     max_height_km : float
#         最大搜索高度 (km)
    
#     Returns:
#     --------
#     dict with xr.DataArray:
#         - tropopause_height: 对流层顶高度 (m), shape: (time, lat, lon)
#         - tropopause_temperature: 对流层顶温度 (K), shape: (time, lat, lon)
#         - tropopause_pressure: 对流层顶气压 (Pa), shape: (time, lat, lon)
#         - tropopause_level_index: 对流层顶层次索引, shape: (time, lat, lon)
#     """
    
#     print("开始计算对流层顶...")
#     print(f"  数据形状: {ta_data.shape}")
#     print(f"  搜索范围: {min_height_km} - {max_height_km} km")
    
#     # 转换高度到km
#     height_km = height_data / 1000.0
    
#     # 创建高度掩码（只在指定范围内搜索）
#     height_mask = (height_km >= min_height_km) & (height_km <= max_height_km)
#     valid_levels = np.where(height_mask)[0]
    
#     if len(valid_levels) == 0:
#         raise ValueError(f"没有层次在指定高度范围内 ({min_height_km}-{max_height_km} km)")
    

#     # 只选择有效高度范围的数据
#     ta_subset = ta_data.isel({lev_dim: valid_levels})
   
#     height_subset = height_data.isel({lev_dim: valid_levels})
    
#     # 使用 xarray 的 argmin 找到温度最小值的索引（沿着lev维度）
#     # 返回的是相对于 subset 的索引
#     trop_idx_subset = ta_subset.argmin(dim=lev_dim)
    
#     # 转换为原始数据的绝对索引
#     trop_idx_absolute = xr.DataArray(
#         valid_levels[trop_idx_subset.values],
#         coords=trop_idx_subset.coords,
#         dims=trop_idx_subset.dims
#     )
    
#     # 使用高级索引提取对流层顶处的值
#     # 创建索引数组
#     time_idx = xr.DataArray(np.arange(ta_data.sizes['time']), dims=['time'])
#     lat_idx = xr.DataArray(np.arange(ta_data.sizes[lat_dim]), dims=[lat_dim])
#     lon_idx = xr.DataArray(np.arange(ta_data.sizes[lon_dim]), dims=[lon_dim])
    
#     # 提取对流层顶的温度、气压和高度
#     # 使用 isel 和索引
#     trop_temperature = ta_data.isel({
#         lev_dim: trop_idx_absolute
#     })
    
#     trop_height = height_data.isel({
#         lev_dim: trop_idx_absolute
#     })
    
#     # 添加属性
#     trop_height.attrs = {
#         'long_name': 'Tropopause Height',
#         'units': 'm',
#         'method': 'Temperature Minimum'
#     }
    
#     trop_temperature.attrs = {
#         'long_name': 'Tropopause Temperature',
#         'units': 'K'
#     }
    
    
#     trop_idx_absolute.attrs = {
#         'long_name': 'Tropopause Level Index',
#         'units': '1'
#     }
    
#     trop_idx_absolute.attrs = {
#         'long_name': 'Tropopause Level Index',
#         'units': '1'
#     }
    
#     result = {
#         'tropopause_height': trop_height,
#         'tropopause_temperature': trop_temperature,
#         'tropopause_level_index': trop_idx_absolute
#     }
    
#     print("✓ 对流层顶计算完成")
    
#     return result

# # ============================================================================
# # 对三个实验进行计算
# # ============================================================================

# print("="*90)
# print("对流层顶计算 - 基于温度最小值方法（空间场）")
# print("="*90)

# # 准备数据：选择热带海洋区域
# print("\n准备输入数据...")
# ta_cntl_tropical = ta_cntl[ta_var].sel({lat_dim: slice(TROPICAL_BOUNDS[0], TROPICAL_BOUNDS[1])})
# ta_p4k_tropical = ta_p4k[ta_var].sel({lat_dim: slice(TROPICAL_BOUNDS[0], TROPICAL_BOUNDS[1])})
# ta_4co2_tropical = ta_4co2[ta_var].sel({lat_dim: slice(TROPICAL_BOUNDS[0], TROPICAL_BOUNDS[1])})


# # 高度数据（从 zg_cntl_km，已经是 km 单位）
# # 需要转换回米以匹配函数期望
# if hasattr(zg_cntl_km, 'metpy'):
#     # MetPy Quantity - 提取数值
#     height_1d_values = zg_cntl_km.metpy.dequantify().values * 1000.0  # km -> m
# else:
#     height_1d_values = zg_cntl_km.values * 1000.0  # km -> m

# # 创建xarray DataArray
# height_1d = xr.DataArray(
#     height_1d_values,
#     coords={lev_dim: zg_cntl_km[lev_dim]},
#     dims=[lev_dim],
#     attrs={'units': 'm', 'long_name': 'Height'}
# )

# print(f"  ta_cntl_tropical shape: {ta_cntl_tropical.shape}")
# print(f"  height_1d shape: {height_1d.shape}")
# print(f"  height_1d range: {float(height_1d.min()):.1f} - {float(height_1d.max()):.1f} m")

# # 计算 CNTL
# print("\n" + "-"*90)
# print("计算 CNTL...")
# print("-"*90)
# trop_field_cntl = calculate_tropopause_from_temperature_minimum(
#     ta_cntl_tropical, height_1d,
#     min_height_km=0, max_height_km=25
# )

# # 计算 P4K
# print("\n" + "-"*90)
# print("计算 P4K...")
# print("-"*90)
# trop_field_p4k = calculate_tropopause_from_temperature_minimum(
#     ta_p4k_tropical, height_1d,
#     min_height_km=0, max_height_km=25
# )

# # 计算 4CO2
# print("\n" + "-"*90)
# print("计算 4CO2...")
# print("-"*90)
# trop_field_4co2 = calculate_tropopause_from_temperature_minimum(
#     ta_4co2_tropical, height_1d,
#     min_height_km=0, max_height_km=25
# )

# print("\n" + "="*90)
# print("✓ 所有实验计算完成")
# print("="*90)

### 9.1 统计分析与空间分布 (Statistical Analysis and Spatial Distribution)

In [11]:
# """
# 统计分析对流层顶的空间分布特征
# """

# # 计算时间平均
# print("计算时间平均...")
# trop_height_cntl_tmean = trop_field_cntl['tropopause_height'].mean(dim='time')
# trop_height_p4k_tmean = trop_field_p4k['tropopause_height'].mean(dim='time')
# trop_height_4co2_tmean = trop_field_4co2['tropopause_height'].mean(dim='time')

# trop_temp_cntl_tmean = trop_field_cntl['tropopause_temperature'].mean(dim='time')
# trop_temp_p4k_tmean = trop_field_p4k['tropopause_temperature'].mean(dim='time')
# trop_temp_4co2_tmean = trop_field_4co2['tropopause_temperature'].mean(dim='time')

# # 计算差异
# trop_height_diff_p4k = trop_height_p4k_tmean - trop_height_cntl_tmean
# trop_height_diff_4co2 = trop_height_4co2_tmean - trop_height_cntl_tmean

# # 应用海洋mask
# ocean_mask_tropical = ocean_mask.sel({lat_dim: slice(TROPICAL_BOUNDS[0], TROPICAL_BOUNDS[1])})

# # 仅海洋点统计
# trop_height_cntl_ocean = trop_height_cntl_tmean.where(ocean_mask_tropical).values.flatten()
# trop_height_p4k_ocean = trop_height_p4k_tmean.where(ocean_mask_tropical).values.flatten()
# trop_height_4co2_ocean = trop_height_4co2_tmean.where(ocean_mask_tropical).values.flatten()

# # 移除 NaN
# trop_height_cntl_ocean = trop_height_cntl_ocean[~np.isnan(trop_height_cntl_ocean)]
# trop_height_p4k_ocean = trop_height_p4k_ocean[~np.isnan(trop_height_p4k_ocean)]
# trop_height_4co2_ocean = trop_height_4co2_ocean[~np.isnan(trop_height_4co2_ocean)]

# print("\n" + "="*90)
# print("对流层顶高度统计（海洋格点，基于温度最小值）")
# print("="*90)

# print(f"\n{'Experiment':<12} {'Mean (km)':<12} {'Std (km)':<12} {'Min (km)':<12} {'Max (km)':<12} {'N points':<12}")
# print("-"*90)

# for exp_name, data in [('CNTL', trop_height_cntl_ocean),
#                         ('P4K', trop_height_p4k_ocean),
#                         ('4CO2', trop_height_4co2_ocean)]:
#     print(f"{exp_name:<12} {np.mean(data)/1000:<12.3f} {np.std(data)/1000:<12.3f} "
#           f"{np.min(data)/1000:<12.3f} {np.max(data)/1000:<12.3f} {len(data):<12}")

# print("\n变化量:")
# print("-"*90)
# mean_change_p4k = np.mean(trop_height_p4k_ocean) - np.mean(trop_height_cntl_ocean)
# mean_change_4co2 = np.mean(trop_height_4co2_ocean) - np.mean(trop_height_cntl_ocean)
# pct_change_p4k = (mean_change_p4k / np.mean(trop_height_cntl_ocean)) * 100
# pct_change_4co2 = (mean_change_4co2 / np.mean(trop_height_cntl_ocean)) * 100

# print(f"P4K - CNTL:  {mean_change_p4k:+.1f} m  ({mean_change_p4k/1000:+.3f} km)  [{pct_change_p4k:+.2f}%]")
# print(f"4CO2 - CNTL: {mean_change_4co2:+.1f} m  ({mean_change_4co2/1000:+.3f} km)  [{pct_change_4co2:+.2f}%]")
# print("="*90)

### 9.4 对流层顶空间分布图 (Spatial Distribution of Tropopause Height)

In [12]:

# # 投影
# proj = ccrs.PlateCarree(180)

# fig, axes = plt.subplots(3, 1, figsize=(12, 9), subplot_kw={'projection': proj}, constrained_layout=True)

# datasets = [
#     (trop_height_cntl_tmean, '(a) CNTL Tropopause Height'),
#     (trop_height_p4k_tmean, '(b) P4K Tropopause Height'),
#     (trop_height_4co2_tmean, '(c) 4CO2 Tropopause Height')
# ]

# # 创建网格 (一次)
# lats = trop_height_cntl_tmean[lat_dim].values
# lons = trop_height_cntl_tmean[lon_dim].values
# LON, LAT = np.meshgrid(lons, lats)

# for ax, (data, title) in zip(axes, datasets):
#     cf = ax.pcolormesh(LON, LAT, data.where(_ocean(data))/1000, 
#                        cmap=cmaps.WhiteBlueGreenYellowRed,
#                        transform=ccrs.PlateCarree(),
#                        shading='auto')
#     ax.coastlines(linewidth=0.5)
#     ax.add_feature(cfeature.BORDERS, linewidth=0.3)
#     ax.set_title(title, fontsize=13,loc='left')
    
#     gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
#     gl.top_labels = False
#     gl.right_labels = False

# # 共享 colorbar
# cbar = fig.colorbar(cf, ax=axes, orientation='horizontal', pad=0.05, aspect=40, shrink=0.5)
# cbar.set_label('Height (km)', fontsize=11)

# plt.savefig(fig_save_dir + 'tropopause_height_maps.png', dpi=300, bbox_inches='tight')
# plt.show()

In [13]:
# ============================================================================
# 第二行：对流层顶高度变化 (m)
# ============================================================================
# P4K - CNTL
# ax4 = plt.subplot(2, 3, 5, projection=projection)
# cf4 = ax4.pcolormesh(LON, LAT, trop_height_diff_p4k.values,
#                      vmin=0, vmax=3000, cmap='RdBu_r',
#                      transform=projection, shading='auto')
# ax4.coastlines(linewidth=0.5)
# ax4.add_feature(cfeature.BORDERS, linewidth=0.3)
# ax4.set_title('(d) P4K - CNTL', fontsize=13, fontweight='bold')
# ax4.set_extent([lons.min(), lons.max(), lats.min(), lats.max()], crs=projection)
# gl4 = ax4.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
# gl4.top_labels = False
# gl4.right_labels = False
# cbar4 = plt.colorbar(cf4, ax=ax4, orientation='horizontal', pad=0.05, aspect=40)
# cbar4.set_label('Δ Height (m)', fontsize=11, fontweight='bold')

# # 4CO2 - CNTL
# ax5 = plt.subplot(2, 3, 6, projection=projection)
# cf5 = ax5.pcolormesh(LON, LAT, trop_height_diff_4co2.values,
#                      vmin=0, vmax=400, cmap='RdBu_r',
#                      transform=projection, shading='auto')
# ax5.coastlines(linewidth=0.5)
# ax5.add_feature(cfeature.BORDERS, linewidth=0.3)
# ax5.set_title('(e) 4CO2 - CNTL', fontsize=13, fontweight='bold')
# ax5.set_extent([lons.min(), lons.max(), lats.min(), lats.max()], crs=projection)
# gl5 = ax5.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
# gl5.top_labels = False
# gl5.right_labels = False
# cbar5 = plt.colorbar(cf5, ax=ax5, orientation='horizontal', pad=0.05, aspect=40)
# cbar5.set_label('Δ Height (m)', fontsize=11, fontweight='bold')

